# M2.8 · GMM ve EM

Yumuşak kümeleme ve gizli değişkenli modellerin genel algoritması. Üç noktada bir E ve M adımını elle atıyor, saf Python'da 1-boyutlu EM'i yazıyor, NumPy'da tam kovaryanslı GMM'yi einsum ile döngüsüz kuruyor, PyTorch'ta aynı modeli `MixtureSameFamily` ile gradyanla eğitip EM ile karşılaştırıyor ve en sonda R başlatmalı EM'i tek tensörde koşturuyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m2-8-gmm-em.html](https://m-gonul.github.io/pytorch-lab/m2-8-gmm-em.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m2_8_gmm_em.py
CHECKS = {
 "b": "import math\nimport random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\", \"sklearn\"}))\n\n    def es():\n        Rm = need(ns, \"e_step\")([1, 2, 4], [0.5, 0.5], [1, 4], [1, 1])\n        close([r[0] for r in Rm], [0.989013, 0.817574, 0.010987], atol=1e-5)\n        close([sum(r) for r in Rm], [1, 1, 1], what=\"satır toplamları\")\n        Rf = need(ns, \"e_step\")([500.0], [0.5, 0.5], [0, 1], [1, 1])\n        ensure(all(math.isfinite(v) for v in Rf[0]), \"uzak noktada NaN; log uzayında çalış\")\n        close(Rf[0], [0, 1], atol=1e-9, what=\"x = 500\")\n    R.test(\"e_step: elle örnek ve uzak noktada kararlı\", es)\n\n    def ms():\n        Rm = need(ns, \"e_step\")([1, 2, 4], [0.5, 0.5], [1, 4], [1, 1])\n        pis, mus, vs = need(ns, \"m_step\")([1, 2, 4], Rm)\n        close([pis[0], mus[0]], [0.605858, 1.467978], atol=2e-5)\n    R.test(\"m_step: π₁ = 0.606, μ₁ = 1.468\", ms)\n\n    def mono():\n        rng = random.Random(0)\n        xs = [rng.gauss(-2, 0.7) for _ in range(60)] + [rng.gauss(3, 1.2) for _ in range(90)]\n        pis, mus, vs, hist = need(ns, \"em\")(xs, [0.5, 0.5], [0.0, 1.0], [1.0, 1.0], 30)\n        ensure(len(hist) == 31, f\"ll_geçmişi uzunluğu {len(hist)}, 31 bekleniyordu\")\n        ensure(all(b >= a - 1e-9 for a, b in zip(hist, hist[1:])), \"log-likelihood bir iterasyonda düştü\")\n        o = sorted(range(2), key=lambda k: mus[k])\n        close([mus[o[0]], mus[o[1]]], [-2, 3], atol=0.35, rtol=0, what=\"bulunan ortalamalar\")\n        return f\"ll: {hist[0]:.1f} → {hist[-1]:.1f}; μ = {mus[o[0]]:.2f}, {mus[o[1]]:.2f}\"\n    R.test(\"em: log-likelihood monoton, ortalamaları buluyor\", mono)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, no_imports, need, ensure\n\n\ndef ref_logpdf(x, mu, S):\n    d = len(x); diff = x - mu\n    return -0.5 * (d * np.log(2 * np.pi) + np.log(np.linalg.det(S)) + diff @ np.linalg.solve(S, diff))\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    for fn in (\"log_gauss\", \"e_step_np\", \"m_step_np\"):\n        R.test(f\"{fn}: döngü yok\", lambda fn=fn: max_loops(src, 0, func=fn))\n    R.test(\"em_np: yalnız iterasyon döngüsü\", lambda: max_loops(src, 1, func=\"em_np\"))\n    R.test(\"torch/sklearn/scipy kullanılmıyor\", lambda: no_imports(src, {\"torch\", \"sklearn\", \"scipy\"}))\n    rng = np.random.default_rng(0)\n    X = np.concatenate([rng.multivariate_normal([0, 0, 0], [[1, .5, 0], [.5, 1, .2], [0, .2, 1]], 120), rng.normal([4, 1, -2], 0.6, (80, 3))])\n    mus = X[[0, 150]]\n    covs = np.stack([np.eye(3) * 2, np.array([[1, .3, 0], [.3, 1, 0], [0, 0, .5]])])\n\n    def lg():\n        L = need(ns, \"log_gauss\")(X, mus, covs)\n        ensure(np.asarray(L).shape == (200, 2), f\"shape {np.asarray(L).shape}\")\n        close(L[[0, 7, 199]], [[ref_logpdf(X[i], mus[k], covs[k]) for k in range(2)] for i in (0, 7, 199)])\n    R.test(\"log_gauss tek tek yoğunlukla aynı\", lg)\n\n    def ms():\n        Rr, _ = need(ns, \"e_step_np\")(X, np.array([0.3, 0.7]), mus, covs)\n        close(Rr.sum(1), np.ones(200), what=\"responsibility satır toplamı\")\n        p, m, C = need(ns, \"m_step_np\")(X, Rr, reg=0.0)\n        for k in range(2):\n            w = Rr[:, k]\n            mk = (w[:, None] * X).sum(0) / w.sum()\n            Ck = ((X - mk).T * w) @ (X - mk) / w.sum()\n            close(m[k], mk, what=f\"μ_{k}\"); close(C[k], Ck, what=f\"Σ_{k}\")\n    R.test(\"e_step_np + m_step_np: ağırlıklı MLE\", ms)\n\n    def em():\n        p, m, C, lls = need(ns, \"em_np\")(X, np.array([0.5, 0.5]), mus, covs, 25)\n        lls = np.asarray(lls)\n        ensure(np.all(np.diff(lls) >= -1e-8), \"log-likelihood düştü\")\n        return f\"ll: {lls[0]:.1f} → {lls[-1]:.1f}\"\n    R.test(\"em_np: log-likelihood monoton\", em)\n",
 "d": "import math\nimport torch\nfrom labcheck import close, need, ensure\n\n\ndef em_ref(X, mus, iters=60):\n    n, d = X.shape; K = mus.shape[0]\n    pis = torch.full((K,), 1.0 / K, dtype=X.dtype); covs = torch.eye(d, dtype=X.dtype).repeat(K, 1, 1)\n    for _ in range(iters):\n        diff = X[:, None] - mus[None]\n        maha = torch.einsum(\"nkd,kde,nke->nk\", diff, torch.linalg.inv(covs), diff)\n        L = pis.log() - 0.5 * (d * math.log(2 * math.pi) + torch.logdet(covs) + maha)\n        Rr = (L - torch.logsumexp(L, 1, keepdim=True)).exp()\n        Nk = Rr.sum(0); pis = Nk / n; mus = Rr.T @ X / Nk[:, None]\n        diff = X[:, None] - mus[None]\n        covs = torch.einsum(\"nk,nkd,nke->kde\", Rr, diff, diff) / Nk[:, None, None] + 1e-6 * torch.eye(d, dtype=X.dtype)\n    diff = X[:, None] - mus[None]\n    L = pis.log() - 0.5 * (d * math.log(2 * math.pi) + torch.logdet(covs) + torch.einsum(\"nkd,kde,nke->nk\", diff, torch.linalg.inv(covs), diff))\n    return float(torch.logsumexp(L, 1).mean())\n\n\ndef check(ns, R):\n    g = torch.Generator().manual_seed(0)\n    X = torch.cat([torch.randn(150, 2, generator=g, dtype=torch.float64) @ torch.tensor([[1.0, 0.8], [0, 0.6]], dtype=torch.float64),\n                   torch.randn(100, 2, generator=g, dtype=torch.float64) * 0.5 + torch.tensor([4.0, 0.0], dtype=torch.float64)])\n\n    def st():\n        raw = torch.randn(3, 2, 2, generator=g, dtype=torch.float64)\n        L = need(ns, \"make_scale_tril\")(raw)\n        ensure(torch.equal(L, torch.tril(L)), \"alt üçgen değil\")\n        ensure(bool((torch.diagonal(L, dim1=-2, dim2=-1) > 0).all()), \"köşegen pozitif değil\")\n    R.test(\"make_scale_tril: alt üçgen, pozitif köşegen\", st)\n\n    def ll():\n        logits = torch.tensor([0.3, -0.2], dtype=torch.float64)\n        mus = torch.tensor([[0.0, 0.0], [4.0, 0.0]], dtype=torch.float64)\n        raw = torch.randn(2, 2, 2, generator=g, dtype=torch.float64) * 0.3\n        Ls = torch.tril(raw, -1) + torch.diag_embed(torch.diagonal(raw, dim1=-2, dim2=-1).exp())\n        covs = Ls @ Ls.transpose(-1, -2)\n        diff = X[:, None] - mus[None]\n        Lg = torch.log_softmax(logits, 0) - 0.5 * (2 * math.log(2 * math.pi) + torch.logdet(covs) + torch.einsum(\"nkd,kde,nke->nk\", diff, torch.linalg.inv(covs), diff))\n        close(need(ns, \"gmm_mean_ll\")(X, logits, mus, raw), torch.logsumexp(Lg, 1).mean())\n    R.test(\"MixtureSameFamily = elle formül\", ll)\n\n    def fit():\n        mus0 = X[[0, 200]].clone()\n        v = need(ns, \"fit_gmm_gd\")(X, mus0)\n        ref = em_ref(X, mus0)\n        ensure(isinstance(v, float), \"float döndürmeli\")\n        ensure(v > ref - 0.02, f\"gradyan ile ortalama ll {v:.4f}, EM {ref:.4f}\")\n        return f\"ortalama log-likelihood: gradyan {v:.4f}, EM (60 iterasyon) {ref:.4f}\"\n    R.test(\"gradyanla eğitim EM'in çözümüne ulaşıyor\", fit)\n",
 "e": "import math\nimport torch\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef one(X, mus, iters, reg=1e-6):\n    n, d = X.shape; K = mus.shape[0]\n    pis = torch.full((K,), 1.0 / K, dtype=X.dtype); vs = X.var(0, unbiased=False).repeat(K, 1)\n    lp = lambda pis, mus, vs: pis.log() - 0.5 * (torch.log(2 * math.pi * vs)[None] + (X[:, None] - mus[None]) ** 2 / vs[None]).sum(-1)\n    for _ in range(iters):\n        L = lp(pis, mus, vs)\n        Rr = (L - torch.logsumexp(L, 1, keepdim=True)).exp()\n        Nk = Rr.sum(0); pis = Nk / n; mus = Rr.T @ X / Nk[:, None]\n        vs = (Rr[..., None] * (X[:, None] - mus[None]) ** 2).sum(0) / Nk[:, None] + reg\n    return torch.logsumexp(lp(pis, mus, vs), 1).sum(), mus\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"yalnız iterasyon döngüsü\", lambda: max_loops(src, 1, func=\"em_restarts_diag\", why=\"başlatmalar üzerinde döngü olmamalı\"))\n    g = torch.Generator().manual_seed(0)\n    X = torch.cat([torch.randn(60, 2, generator=g, dtype=torch.float64) + torch.tensor(c, dtype=torch.float64) for c in ([0.0, 0], [5, 0], [0, 5])])\n    mus0 = X[torch.randint(0, 180, (8, 3), generator=g)]\n\n    def eq():\n        lls, (p, m, v) = need(ns, \"em_restarts_diag\")(X, mus0, 25)\n        ensure(tuple(lls.shape) == (8,), f\"lls shape {tuple(lls.shape)}\")\n        ref = [one(X, mus0[r], 25) for r in range(8)]\n        close(lls, torch.stack([a for a, _ in ref]), rtol=1e-8)\n        close(m, ref[int(lls.argmax())][1], rtol=1e-8, what=\"en iyi başlatmanın ortalamaları\")\n        return f\"ll aralığı {float(lls.min()):.1f} – {float(lls.max()):.1f}\"\n    R.test(\"8 başlatma: tek tek EM ile aynı\", eq)\n    if \"em_restarts_diag\" not in ns:\n        return\n    Xb = torch.randn(1000, 4, generator=g)\n    mb = Xb[torch.randint(0, 1000, (32, 5), generator=g)]\n    t_loop = bench(lambda: [one(Xb, mb[r], 20) for r in range(32)], repeat=3, number=1)\n    t_vec = bench(lambda: ns[\"em_restarts_diag\"](Xb, mb, 20), repeat=3, number=1)\n    R.table(\"32 başlatma × 20 EM iterasyonu (n = 1000, K = 5)\", [\"yöntem\", \"süre\", \"döngüye göre\"],\n            [[\"for r: EM\", fmt_time(t_loop), \"1×\"], [\"senin em_restarts_diag\", fmt_time(t_vec), f\"{t_loop / t_vec:,.1f}× hızlı\"]])\n"
}

In [ ]:
import importlib, labcheck, checks_m2_8_gmm_em as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Gauss karışım modeli (GMM) her noktanın gizli bir bileşenden $z\in\{1..K\}$ geldiğini varsayar:

$$
p(x)=\sum_{k=1}^K\pi_k\,\mathcal N(x\mid\mu_k,\Sigma_k),\qquad
\ell(\theta)=\sum_i\log\sum_k\pi_k\,\mathcal N(x_i\mid\mu_k,\Sigma_k).
$$

Toplamın logaritması kapalı form MLE'yi engeller. **EM** iki kolay adımı tekrarlar:

- **E:** her nokta için bileşenlerin posterior olasılığı, responsibility: $r_{ik}=\frac{\pi_k\mathcal N(x_i\mid\mu_k,\Sigma_k)}{\sum_j\pi_j\mathcal N(x_i\mid\mu_j,\Sigma_j)}$.
- **M:** $r$'yi ağırlık sayarak her bileşen için ağırlıklı MLE:

$$
N_k=\sum_ir_{ik},\quad \pi_k=\frac{N_k}{n},\quad \mu_k=\frac{1}{N_k}\sum_ir_{ik}x_i,\quad \Sigma_k=\frac1{N_k}\sum_ir_{ik}(x_i-\mu_k)(x_i-\mu_k)^\top .
$$

Sayısal olarak her şey log uzayında yapılır: $\log r_{ik}=\log\pi_k+\log\mathcal N_{ik}-\text{LSE}_j(\cdot)$.

## 1 · Elle

<!-- çözüm -->

Üç noktayı ve iki çan eğrisini çiz. $x=1$ neredeyse tamamen 1. bileşene, $x=4$ neredeyse tamamen 2. bileşene ait. $x=2$ arada kalıyor ama 1. bileşene daha yakın: responsibility $0.82$.

M adımında 1. bileşenin ortalaması $1$'den $1.47$'ye kayıyor: $x=2$ onu sağa çekti. Ağırlık $\pi_1$ $0.5$'ten $0.61$'e çıkıyor, çünkü noktaların ~$1.82$'si bu bileşene "ait".

Tablo: $e^{-0.5}\approx0.6065$, $e^{-2}\approx0.1353$, $e^{-4.5}\approx0.01111$.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**$x=[1,2,4]$; iki bileşen $\mu=(1,4)$, $\sigma^2=(1,1)$, $\pi=(0.5,0.5)$. E adımı: $r_{i1}=\frac{\pi_1\mathcal N(x_i\mid\mu_1)}{\sum_k\pi_k\mathcal N(x_i\mid\mu_k)}$**

- `r11`: $r_{11}$ ($x=1$)
- `r21`: $r_{21}$ ($x=2$)
- `r31`: $r_{31}$ ($x=4$)

**M adımı (bileşen 1)**

- `N1`: $N_1=\sum_i r_{i1}$
- `pi1`: $\pi_1=N_1/n$
- `mu1`: $\mu_1=\frac1{N_1}\sum_i r_{i1}x_i$

In [ ]:
cevaplar = dict(
    r11=None,
    r21=None,
    r31=None,
    N1=None,
    pi1=None,
    mu1=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"r11":[0.989013,0.002],"r21":[0.817574,0.002],"r31":[0.010987,0.002],"N1":[1.817574,0.003],"pi1":[0.605858,0.002],"mu1":[1.467978,0.003]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, 1-boyutlu GMM. Parametreler listeler: `pis`, `mus`, `vars` (her biri uzunluk $K$).

- `e_step(xs, pis, mus, vars)` → `R` (n × K liste); log uzayında hesapla, satır başına max çıkararak normalize et.
- `m_step(xs, R, reg=1e-6)` → `(pis, mus, vars)`; varyansa `reg` ekle.
- `log_likelihood(xs, pis, mus, vars)` — log-sum-exp ile.
- `em(xs, pis, mus, vars, iters)` → `(pis, mus, vars, ll_geçmişi)`; `ll_geçmişi[0]` başlangıç parametrelerinin, sonrakiler her iterasyondan sonraki log-likelihood.

In [ ]:
%%writefile stage_b.py
import math


def e_step(xs, pis, mus, vars):
    ...


def m_step(xs, R, reg=1e-6):
    ...


def log_likelihood(xs, pis, mus, vars):
    ...


def em(xs, pis, mus, vars, iters):
    ...


R = e_step([1, 2, 4], [0.5, 0.5], [1, 4], [1, 1])
print([round(r[0], 4) for r in R])          # [0.989, 0.8176, 0.011]
print(m_step([1, 2, 4], R))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

$d$ boyut, tam kovaryans, $K$ bileşen; **döngü yok** (iterasyon döngüsü hariç).

Log-yoğunluk tüm $(i,k)$ çiftleri için:

$$
\log\mathcal N_{ik}=-\tfrac12\Big(d\log2\pi+\log\det\Sigma_k+(x_i-\mu_k)^\top\Sigma_k^{-1}(x_i-\mu_k)\Big).
$$

`diff = X[:, None] - mus[None]` `(n, K, d)`, `P = np.linalg.inv(covs)` `(K, d, d)`, Mahalanobis `np.einsum("nkd,kde,nke->nk", diff, P, diff)`, `np.linalg.slogdet(covs)[1]` `(K,)`.

- `log_gauss(X, mus, covs)` → `(n, K)`.
- `e_step_np(X, pis, mus, covs)` → `(R: (n, K), ll: float)`.
- `m_step_np(X, R, reg=1e-6)` → `(pis, mus, covs)`; kovaryans `np.einsum("nk,nkd,nke->kde", R, diff, diff) / Nk[:, None, None]` + `reg·I`.
- `em_np(X, pis, mus, covs, iters)` → `(pis, mus, covs, lls)`.

In [ ]:
%%writefile stage_c.py
import numpy as np


def log_gauss(X, mus, covs):
    """→ (n, K). Döngü yok."""
    ...


def e_step_np(X, pis, mus, covs):
    ...


def m_step_np(X, R, reg=1e-6):
    ...


def em_np(X, pis, mus, covs, iters):
    ...


rng = np.random.default_rng(0)
X = np.concatenate([rng.multivariate_normal([0, 0], [[1, 0.8], [0.8, 1]], 150), rng.normal([4, 0], 0.5, (100, 2))])
pis, mus, covs, lls = em_np(X, np.array([0.5, 0.5]), X[[0, -1]], np.stack([np.eye(2)] * 2), 30)
print(mus, lls[-1])

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

EM'e alternatif: log-likelihood'u doğrudan gradyanla maksimize et. Kısıtlar yeniden parametrelendirmeyle çözülür: $\pi=\text{softmax}(a)$, $\Sigma_k=L_kL_k^\top$, $L_k$ alt üçgen ve köşegeni $e^{(\cdot)}$ ile pozitif.

```python
from torch.distributions import Categorical, MultivariateNormal, MixtureSameFamily
gmm = MixtureSameFamily(Categorical(logits=a), MultivariateNormal(mus, scale_tril=L))
ll = gmm.log_prob(X).sum()
```

- `make_scale_tril(raw)` — `raw: (K, d, d)` → alt üçgen, köşegeni `exp(diag)`: `torch.tril(raw, -1) + torch.diag_embed(torch.diagonal(raw, dim1=-2, dim2=-1).exp())`.
- `gmm_mean_ll(X, logits, mus, raw)` → ortalama log-likelihood.
- `fit_gmm_gd(X, mus0, steps=800, lr=0.05)` → son ortalama log-likelihood (float); `logits` ve `raw` sıfırdan, `Adam`.

Kontrol, `MixtureSameFamily`'nin c'deki formülünle aynı sayıyı verdiğini ve gradyanla bulunan çözümün EM'inkine yakın olduğunu doğrular.

In [ ]:
%%writefile stage_d.py
import torch
from torch.distributions import Categorical, MultivariateNormal, MixtureSameFamily


def make_scale_tril(raw):
    ...


def gmm_mean_ll(X, logits, mus, raw):
    ...


def fit_gmm_gd(X, mus0, steps=800, lr=0.05):
    """→ son ortalama log-likelihood (float)."""
    ...

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

EM yerel maksimuma takılır; çare k-means'teki gibi çoklu başlatma. $R$ başlatmayı bir batch boyutu yap. Hesabı basit tutmak için **diyagonal** kovaryans: $\log\mathcal N_{rik}=-\frac12\sum_j\big(\log2\pi\sigma^2_{rkj}+\frac{(x_{ij}-\mu_{rkj})^2}{\sigma^2_{rkj}}\big)$, tensör `(R, n, K)`.

- `em_restarts_diag(X, mus0, iters, reg=1e-6)` — `mus0: (R, K, d)`; başlangıçta `pis` eşit, varyanslar her başlatmada `X.var(0, unbiased=False)`. → `(lls: (R,), best: (pis, mus, vars))`. Yalnız iterasyon döngüsü.

Kontrol, her başlatmayı ayrı ayrı koşturan referansla karşılaştırır.

In [ ]:
%%writefile stage_e.py
import math
import torch


def em_restarts_diag(X, mus0, iters, reg=1e-6):
    """mus0: (R, K, d) → (lls: (R,), best: (pis, mus, vars)). Yalnız iterasyon döngüsü."""
    ...


g = torch.Generator().manual_seed(0)
X = torch.cat([torch.randn(100, 2, generator=g) + torch.tensor(c) for c in ([0.0, 0], [5, 0], [0, 5])])
mus0 = X[torch.randint(0, 300, (16, 3), generator=g)]
lls, best = em_restarts_diag(X, mus0, 30)
print(lls)

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Makale ödevi: Dempster, Laird, Rubin (1977): EM algoritması

[Dempster, Laird, Rubin — Maximum Likelihood from Incomplete Data via the EM Algorithm (JRSS-B 1977)](https://www.jstor.org/stable/2984875)

Makalenin monoton artış teoremini (Teorem 1) Gauss karışımında her yinelemede log-olabilirliği kaydederek doğrula; eksik veri yorumuyla eksik değerli çok değişkenli normal tahminini (makalenin Bölüm 4.1.3 örneği) EM ile yaz. Yakınsama hızının eksik bilgi oranına bağlı olduğunu (Bölüm 3) iki bileşen ayrılığı değişirken yinelenme sayısını ölçerek göster.


**Adımlar**

1. Eksik veri: rastgele %20 bileşeni gizle, E-adımında koşullu beklentileri kullan.

**Kabul ölçütleri**

- Log-olabilirlik hiçbir yinelemede azalmaz.
- Bileşenler yaklaştıkça EM yavaşlar.

## Kaynaklar

- [Stanford CS229 — Notes: Mixtures of Gaussians and the EM algorithm](https://cs229.stanford.edu/main_notes.pdf) — doğrulanmadı
- [Bishop & Bishop — Deep Learning: Foundations and Concepts, Bölüm 15 (Discrete Latent Variables)](https://www.bishopbook.com/) — doğrulanmadı
- [Murphy — PML1 §8.7 Bound optimization (EM)](https://probml.github.io/pml-book/book1.html) — CC BY-NC-ND
- [PyTorch — torch.distributions.MixtureSameFamily](https://docs.pytorch.org/docs/stable/distributions.html#mixturesamefamily) — BSD-3